# Решения: membership/counts practice

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('orders_slim.csv')
df = pd.read_csv(
    CSV_PATH,
    parse_dates=['order_purchase_timestamp', 'order_estimated_delivery_date', 'order_delivered_customer_date'],
)


In [ ]:
known = set(df['order_id'])
probe_ids = df['order_id'].head(5).tolist() + ['ol_99999']
found = [oid in known for oid in probe_ids]
pair_bucket: dict[tuple[str, str], dict[str, int]] = {}
for r in df[['seller_state', 'customer_state', 'is_late']].itertuples(index=False):
    key = (r.seller_state, r.customer_state)
    if key not in pair_bucket:
        pair_bucket[key] = {'late': 0, 'total': 0}
    pair_bucket[key]['total'] += 1
    pair_bucket[key]['late'] += int(r.is_late)
pair_rate = {k: v['late'] / v['total'] for k, v in pair_bucket.items()}
global_rate = float(df['is_late'].mean())
hot_segments = sorted(
    [(k, round(v, 3)) for k, v in pair_rate.items() if v > global_rate],
    key=lambda x: x[1],
    reverse=True,
)
watch = set(df.sort_values(['delay_days', 'freight_value'], ascending=False)['order_id'].head(10))
SEG_NOTE = (
    'Сегмент полезен, если в нём достаточно наблюдений и устойчивая доля late. '
    'Слишком мелкие сегменты дают шум и могут вести к ложным операционным решениям.'
)
print(found)
print('global_rate=', round(global_rate, 3))
print('hot_segments=', hot_segments[:6])
print('watch=', watch)
print(SEG_NOTE)